In [113]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [114]:
df = pd.read_csv('../data/raw/store-data.csv')

In [115]:
rows, columns = df.shape
print(f'[+] Raw Dataset Contain {rows} Rows And {columns} Columns')

[+] Raw Dataset Contain 10064 Rows And 21 Columns


In [116]:
def get_duplicated_statics():
    duplicatedRows = df.duplicated()
    print(f'[+] {duplicatedRows.sum()} => {(duplicatedRows.mean() * 100):.2f}% of rows are duplicated')
get_duplicated_statics()

[+] 34 => 0.34% of rows are duplicated


In [117]:
def remove_duplicated_columns():
    ids = [id for id, boolean in enumerate(df.duplicated()) if boolean == True]
    df.drop(ids, inplace=True)
    df.reset_index(drop=True, inplace=True)
remove_duplicated_columns()
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='mixed').dt.strftime('%d/%m/%Y')
df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed').dt.strftime('%d/%m/%Y')
remove_duplicated_columns()

In [118]:
get_duplicated_statics()

[+] 0 => 0.00% of rows are duplicated


In [119]:
# changing between ship-date order-date if order-date > ship-date
mask = pd.to_datetime(df['Order Date'], dayfirst=True) > pd.to_datetime(df['Ship Date'], dayfirst=True)
new_orderDateCol = np.where(mask, df['Ship Date'].values, df['Order Date'].values)
new_shipDateCol = np.where(mask, df['Order Date'].values, df['Ship Date'].values)
df['Order Date'] = new_orderDateCol
df['Ship Date'] = new_shipDateCol

In [120]:
def describe_dataset():
    for column in df.columns:
        nullValues = df[column].isnull()
        duplicatedValues = df[column].duplicated().sum()
        print(f'[+] Column "{column}" Contain: {nullValues.sum()} Null Values => {(nullValues.mean() * 100):.2f}% | Duplicated Values: {duplicatedValues}')

In [121]:
rows, _ = df.shape
df['Row ID'] = np.arange(1, rows + 1, 1)

In [122]:
describe_dataset()

[+] Column "Row ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 0
[+] Column "Order ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 5015
[+] Column "Order Date" Contain: 0 Null Values => 0.00% | Duplicated Values: 8786
[+] Column "Ship Date" Contain: 101 Null Values => 1.01% | Duplicated Values: 8690
[+] Column "Ship Mode" Contain: 301 Null Values => 3.00% | Duplicated Values: 10019
[+] Column "Customer ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 9231
[+] Column "Customer Name" Contain: 311 Null Values => 3.10% | Duplicated Values: 9213
[+] Column "Segment" Contain: 0 Null Values => 0.00% | Duplicated Values: 10018
[+] Column "Country" Contain: 0 Null Values => 0.00% | Duplicated Values: 10023
[+] Column "City" Contain: 0 Null Values => 0.00% | Duplicated Values: 9391
[+] Column "State" Contain: 0 Null Values => 0.00% | Duplicated Values: 9938
[+] Column "Postal Code" Contain: 199 Null Values => 1.99% | Duplicated Values: 9318
[+] Column "Region" Contain:

In [123]:
def get_most_close_ship_date(row_id, mean):
    output_arr = np.full(6, mean)
    local_id = row_id
    found_values = 0
    while found_values < 3 and local_id >= 0:
        local_id -= 1
        sub = df[df['Row ID'] == local_id][['Order Date', 'Ship Date']]
        if sub['Ship Date'].isnull().iloc[0]: continue
        output_arr[found_values] = (pd.to_datetime(sub['Ship Date'].iloc[0], dayfirst=True) - pd.to_datetime(sub['Order Date'].iloc[0], dayfirst=True)).total_seconds()
        found_values += 1
    local_id = row_id
    while found_values < 6 and local_id < len(df):
        local_id += 1
        sub = df[df['Row ID'] == local_id][['Order Date', 'Ship Date']]
        if sub['Ship Date'].isnull().iloc[0]: continue
        output_arr[found_values] = (pd.to_datetime(sub['Ship Date'].iloc[0], dayfirst=True) - pd.to_datetime(sub['Order Date'].iloc[0], dayfirst=True)).total_seconds()
        found_values += 1
    return np.divide(output_arr.sum(), 6)



In [124]:
mean_ship_date = np.divide((pd.to_datetime(df['Ship Date'].iloc[0], dayfirst=True) - pd.to_datetime(df['Order Date'].iloc[0], dayfirst=True)).total_seconds(), rows)
for row_id in df['Row ID'][df['Ship Date'].isnull()].values:
    prediction = get_most_close_ship_date(row_id, mean_ship_date)
    target_row = df[df['Row ID'] == row_id]
    order_date = target_row['Order Date'].iloc[0]
    df.loc[target_row.index[0], 'Ship Date'] = pd.to_datetime(pd.to_datetime(order_date, dayfirst=True).timestamp() + prediction).strftime('%d/%m/%Y')

In [125]:
describe_dataset()

[+] Column "Row ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 0
[+] Column "Order ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 5015
[+] Column "Order Date" Contain: 0 Null Values => 0.00% | Duplicated Values: 8786
[+] Column "Ship Date" Contain: 0 Null Values => 0.00% | Duplicated Values: 8690
[+] Column "Ship Mode" Contain: 301 Null Values => 3.00% | Duplicated Values: 10019
[+] Column "Customer ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 9231
[+] Column "Customer Name" Contain: 311 Null Values => 3.10% | Duplicated Values: 9213
[+] Column "Segment" Contain: 0 Null Values => 0.00% | Duplicated Values: 10018
[+] Column "Country" Contain: 0 Null Values => 0.00% | Duplicated Values: 10023
[+] Column "City" Contain: 0 Null Values => 0.00% | Duplicated Values: 9391
[+] Column "State" Contain: 0 Null Values => 0.00% | Duplicated Values: 9938
[+] Column "Postal Code" Contain: 199 Null Values => 1.99% | Duplicated Values: 9318
[+] Column "Region" Contain: 0

In [126]:
get_duplicated_statics()

[+] 0 => 0.00% of rows are duplicated


In [127]:
def get_frequente_ship_mode(row_id, frequente_value):
    output_list = [frequente_value]  * 10
    found_values = 0
    local_row = row_id
    while found_values < 5 and local_row >= 0:
        local_row -= 1
        sub = df['Ship Mode'][df['Row ID'] == local_row]
        if sub.empty: continue
        if pd.isnull(sub.iloc[0]): continue
        output_list[found_values] = sub.iloc[0]
        found_values += 1
    local_row = row_id
    while found_values < 10 and local_row < len(df):
        local_row += 1
        sub = df['Ship Mode'][df['Row ID'] == local_row]
        if sub.empty: continue
        if pd.isnull(sub.iloc[0]): continue
        output_list[found_values] = sub.iloc[0]
        found_values += 1
    return pd.Series(output_list).mode().iloc[0]

In [128]:
mean_ship_mode = df['Ship Mode'].mode().loc[0]
for row_id in df['Row ID'][df['Ship Mode'].isnull()].values:
    prediction = get_frequente_ship_mode(row_id, mean_ship_mode)
    target_row = df[df['Row ID'] == row_id]
    df.loc[target_row.index[0], 'Ship Mode'] = prediction
    # print(f'Order Date: {order_date} | Ship Date (predicted): {pd.to_datetime(pd.to_datetime(order_date, dayfirst=True).timestamp() + prediction, unit='s').strftime('%d/%m/%Y')}')

In [129]:
def clean_customer_name_column():
    customer_id_set = set(df['Customer ID'].values)
    base_customer_name = df['Customer Name'].mode().iloc[0]
    for customer_id in customer_id_set:
        mask = df['Customer ID'] == customer_id
        names = df.loc[mask, 'Customer Name']
        frequente_value = names.mode()
        if frequente_value.empty:
            value = base_customer_name
        else:
            value = frequente_value.iloc[0]
        df.loc[mask, 'Customer Name'] = names.fillna(value)

In [130]:
clean_customer_name_column()

In [131]:
describe_dataset()

[+] Column "Row ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 0
[+] Column "Order ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 5015
[+] Column "Order Date" Contain: 0 Null Values => 0.00% | Duplicated Values: 8786
[+] Column "Ship Date" Contain: 0 Null Values => 0.00% | Duplicated Values: 8690
[+] Column "Ship Mode" Contain: 0 Null Values => 0.00% | Duplicated Values: 10020
[+] Column "Customer ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 9231
[+] Column "Customer Name" Contain: 0 Null Values => 0.00% | Duplicated Values: 9214
[+] Column "Segment" Contain: 0 Null Values => 0.00% | Duplicated Values: 10018
[+] Column "Country" Contain: 0 Null Values => 0.00% | Duplicated Values: 10023
[+] Column "City" Contain: 0 Null Values => 0.00% | Duplicated Values: 9391
[+] Column "State" Contain: 0 Null Values => 0.00% | Duplicated Values: 9938
[+] Column "Postal Code" Contain: 199 Null Values => 1.99% | Duplicated Values: 9318
[+] Column "Region" Contain: 0 Nul

In [132]:
def clean_postal_code_column():
    cities = set(df['City'].values)
    frequente_postal_code = df['Postal Code'].mode().iloc[0]
    for city in cities:
        mask = df['City'] == city
        postal_codes = df.loc[mask, 'Postal Code']
        frequente_value = postal_codes.mode()
        if frequente_value.empty:
            value = frequente_postal_code
        else:
            value = frequente_value.iloc[0]
        df.loc[mask, 'Postal Code'] = postal_codes.fillna(value)

In [133]:
clean_postal_code_column()

In [134]:
df['Initial Product Price'] = np.float32((df['Sales'] - (df['Sales'] / 100 *df['Discount'])) / df['Quantity'])
df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Initial Product Price
0,1,CA-2016-152156,08/11/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136,130.979996
1,2,CA-2016-152156,08/11/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820,243.979996
2,3,CA-2016-138688,12/06/2016,16/06/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714,7.310000
3,4,US-2015-108966,11/10/2015,18/10/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310,190.653687
4,5,US-2015-108966,11/10/2015,18/10/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164,11.161632
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10019,10020,CA-2016-128671,11/08/2016,16/08/2016,Standard Class,MT-18070,MICHELLE TRAN,Home Office,United States,Tulsa,...,Central,OFF-BI-10003007,Office Supplies,Binders,"Premium Transparent Presentation Covers, No Pa...",77.5600,2.0,0.00,35.6776,38.779999
10020,10021,CA-2016-160815,05/09/2016,06/09/2016,First Class,TR-21325,TOBY RITTER,Consumer,United States,Cedar Rapids,...,Central,TEC-PH-10003505,Technology,Phones,Geemarc AmpliPOWER60,278.4000,3.0,0.00,80.7360,92.800003
10021,10022,CA-2017-151799,14/12/2017,18/12/2017,Standard Class,BF-11170,BEN FERRER,Home Office,United States,Lawrence,...,East,OFF-ST-10002790,Office Supplies,Storage,Safco Industrial Shelving,73.8500,1.0,0.00,2.2155,73.849998
10022,10023,CA-2014-111934,05/05/2014,07/05/2014,First Class,GD-14590,GIULIETTA DORTCH,Corporate,United States,Arlington,...,South,OFF-PA-10000474,Office Supplies,Paper,Easy-staple paper,35.4400,1.0,0.00,16.6568,35.439999


In [135]:
def fill_quantity_with_product_mean():
    products_ids = set(df['Product ID'][df['Quantity'].isna()].values)
    for product_id in products_ids:
        mask = df['Product ID'] == product_id
        quantities = df.loc[mask, 'Quantity']
        if quantities.isna().sum() == len(quantities):
            product_sub_category = df[df['Product ID'] == product_id]['Sub-Category'].iloc[0]
            mean = np.int16(df[df['Sub-Category'] == product_sub_category]['Quantity'].mean())
        else: mean = np.int16(quantities.mean())
        df.loc[mask, 'Quantity'] = quantities.fillna(mean)

fill_quantity_with_product_mean()

In [136]:
df[df['Sub-Category'] == 'Paper']['Quantity'].mean()

np.float64(3.7643636363636364)

In [137]:
df[df['Quantity'].isnull()]

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Initial Product Price


In [138]:
describe_dataset()

[+] Column "Row ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 0
[+] Column "Order ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 5015
[+] Column "Order Date" Contain: 0 Null Values => 0.00% | Duplicated Values: 8786
[+] Column "Ship Date" Contain: 0 Null Values => 0.00% | Duplicated Values: 8690
[+] Column "Ship Mode" Contain: 0 Null Values => 0.00% | Duplicated Values: 10020
[+] Column "Customer ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 9231
[+] Column "Customer Name" Contain: 0 Null Values => 0.00% | Duplicated Values: 9214
[+] Column "Segment" Contain: 0 Null Values => 0.00% | Duplicated Values: 10018
[+] Column "Country" Contain: 0 Null Values => 0.00% | Duplicated Values: 10023
[+] Column "City" Contain: 0 Null Values => 0.00% | Duplicated Values: 9391
[+] Column "State" Contain: 0 Null Values => 0.00% | Duplicated Values: 9938
[+] Column "Postal Code" Contain: 0 Null Values => 0.00% | Duplicated Values: 9319
[+] Column "Region" Contain: 0 Null 

In [152]:
def clean_sales_column():
    rows_ids = set(df['Row ID'][df['Sales'].isna()].values)
    for row_id in rows_ids:
        row = df[df['Row ID'] == row_id]
        row_quantity = row['Quantity'].iloc[0]
        by_quantity = df[(df['Product ID'] == row['Product ID'].iloc[0]) & ((np.subtract(row_quantity, row_quantity // 0.3)) <= df['Quantity']) | (df['Quantity'] <= np.sum(row_quantity, row_quantity // 0.3))]['Sales'].mean()
        if by_quantity == np.nan:
            print('is null')
            continue
        print(by_quantity)

clean_sales_column()

TypeError: 'numpy.float64' object cannot be interpreted as an integer

In [141]:
df['Quantity'].max()

np.float64(14.0)

In [ ]:
output['Sales'].iloc[0]

np.float64(51.220000000000006)

In [ ]:
df[df['Discount'] >= 1]

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Initial Product Price
3096,3097,CA-2015-114468,23/08/2015,23/08/2015,Same Day,TD-20995,Tamara Dahlen,Consumer,United States,Bolingbrook,...,Central,OFF-AP-10000696,Office Supplies,Appliances,Holmes Odor Grabber,5.768,2.0,1.5,-13.5548,2.840740
3501,3502,CA-2016-108616,29/09/2016,03/10/2016,Standard Class,JK-15730,Joe Kamberova,Consumer,United States,Mobile,...,South,OFF-AR-10001149,Office Supplies,Art,"Sanford Colorific Colored Pencils, 12/Box",25.920,9.0,1.5,7.7760,2.836800
4551,4552,US-2014-165862,13/07/2014,17/07/2014,Second Class,GK-14620,Grace Kelly,Corporate,United States,Los Angeles,...,West,FUR-TA-10002855,Furniture,Tables,Bevis Round Conference Table Top & Single Colu...,351.216,3.0,1.5,4.3902,115.315918
4710,4711,CA-2017-138149,29/06/2017,30/06/2017,First Class,WB-21850,William Brown,Consumer,United States,Los Angeles,...,West,TEC-AC-10001284,Technology,Accessories,Enermax Briskie RF Wireless Keyboard and Mouse...,62.310,3.0,1.5,22.4316,20.458450
6405,6406,CA-2017-154074,31/08/2017,02/09/2017,Second Class,BW-11110,Bart Watters,Corporate,United States,Spokane,...,West,FUR-CH-10002331,Furniture,Chairs,Hon 4700 Series Mobuis Mid-Back Task Chairs wi...,569.568,2.0,1.5,7.1196,280.512238
6457,6458,CA-2014-110065,05/08/2014,11/08/2014,Standard Class,SP-20650,Stephanie Phelps,Corporate,United States,New York City,...,East,OFF-AR-10004165,Office Supplies,Art,Binney & Smith inkTank Erasable Pocket Highlig...,15.960,7.0,1.5,7.0224,2.245800
6683,6684,CA-2014-109134,05/11/2014,10/11/2014,Standard Class,DE-13255,Deanra Eno,Home Office,United States,Los Angeles,...,West,FUR-FU-10000320,Furniture,Furnishings,OIC Stacking Trays,20.040,6.0,1.5,8.8176,3.289900
8814,8815,CA-2015-144274,23/11/2015,25/11/2015,Second Class,PW-19240,Pierre Wener,Consumer,United States,Wilmington,...,East,TEC-PH-10003012,Technology,Phones,Nortel Meridian M3904 Professional Digital phone,307.980,2.0,1.5,89.3142,151.680145
9291,9292,CA-2014-163650,23/09/2014,25/09/2014,First Class,TT-21220,Thomas Thornton,Consumer,United States,Dover,...,East,OFF-BI-10004965,Office Supplies,Binders,Ibico Covers for Plastic or Wire Binding Elements,34.500,3.0,1.5,15.5250,11.327500
9352,9353,CA-2017-148411,24/09/2017,26/09/2017,First Class,RO-19780,Rose O'Brian,Consumer,United States,Chicago,...,Central,FUR-CH-10003973,furniture,Chairs,GuestStacker Chair with Chrome Finish Legs,520.464,2.0,1.5,-14.8704,256.328522


In [ ]:
df[df['Profit'] < 0]

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Initial Product Price
3,4,US-2015-108966,11/10/2015,18/10/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310,190.653687
14,15,US-2015-118983,22/11/2015,26/11/2015,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,...,Central,OFF-AP-10002311,Office Supplies,Appliances,Holmes Replacement Filter for HEPA Air Cleaner...,68.8100,5.0,0.80,-123.8580,13.651904
15,16,US-2015-118983,22/11/2015,26/11/2015,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,...,Central,OFF-BI-10000756,Office Supplies,Binders,Storex DuraTech Recycled Plastic Frosted Binders,2.5440,3.0,0.80,-3.8160,0.841216
23,24,US-2017-156909,16/07/2017,18/07/2017,Second Class,SF-20065,Sandra Flanagan,Consumer,United States,Philadelphia,...,East,FUR-CH-10002774,Furniture,Chairs,"Global Deluxe Stacking Chair, Gray",71.3720,2.0,0.30,-1.0196,35.578941
27,28,US-2015-150630,17/09/2015,21/09/2015,Standard Class,TB-21520,Tracy Blumstein,Consumer,United States,Philadelphia,...,East,FUR-BO-10004834,Furniture,Bookcases,"Riverside Palais Royal Lawyers Bookcase, Royal...",3083.4300,7.0,0.50,-1665.0522,438.287537
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9998,9999,CA-2017-139311,11/08/2017,13/08/2017,First Class,SF-20965,Sylvia Foulston,Corporate,United States,Bedford,...,Central,OFF-BI-10004209,Office Supplies,Binders,"Fellowes Twister Kit, Gray/Clear, 3/pkg",12.8640,8.0,0.80,-22.5120,1.595136
10002,10003,CA-2016-134222,10/07/2016,11/07/2016,Same Day,NZ-18565,Nick Zandusky,Home Office,United States,YUMA,...,West,OFF-BI-10001658,Office Supplies,Binders,GBC Standard Therm-A-Bind Covers,44.8560,6.0,0.70,-35.8848,7.423668
10003,10004,US-2017-101784,06/07/2017,11/07/2017,Standard Class,PO-18850,Patrick O'Brill,Consumer,United States,Los Angeles,...,West,FUR-CH-10001146,Furniture,Chairs,"Global Task Chair, Black",122.1360,3.0,0.20,-13.7403,40.630577
10006,10007,CA-2016-158099,03/09/2016,05/09/2016,First Class,PK-18910,PAUL KNUTSON,Home Office,United States,Philadelphia,...,East,TEC-PH-10002496,Technology,Phones,Cisco SPA301,280.7820,3.0,0.40,-46.7970,93.219627
